In [1]:
# =======================================================
# Processed E-commerce Dataset (Day 9 Assignment)
# =======================================================

import io
import pandas as pd

# -------------------------------------------------------
# 1. Mock Data Creation (Simulating provided CSV files)
# -------------------------------------------------------
# In actual execution, replace io.StringIO with actual CSV file paths:
# orders_df = pd.read_csv("Orders.csv")
# customers_df = pd.read_csv("Customers.csv")
# products_df = pd.read_csv("Products.csv")

orders_csv = """order_id,customer_id,product_id,order_date,quantity
101,C1,P10,2026-01-15,2
102,C2,P20,2026-02-20,1
103,C1,P30,2026-03-05,4
104,C3,P10,2026-03-12,1
"""

customers_csv = """customer_id,customer_name,customer_segment
C1,Alice,Consumer
C2,Bob,Corporate
C3,Charlie,Consumer
"""

products_csv = """product_id,product_name,category,unit_price
P10,Laptop,Electronics,1200.00
P20,Chair,Furniture,150.00
P30,Headphones,Electronics,80.00
"""

orders_df = pd.read_csv(io.StringIO(orders_csv))
customers_df = pd.read_csv(io.StringIO(customers_csv))
products_df = pd.read_csv(io.StringIO(products_csv))

# -------------------------------------------------------
# 2. Combining DataFrames using merge() and concat()
# -------------------------------------------------------
# Merging Orders with Customers and Products
merged_df = orders_df.merge(customers_df, on="customer_id", how="left")
merged_df = merged_df.merge(products_df, on="product_id", how="left")

# Demonstrating concat() (e.g., merging back supplementary batch orders)
extra_orders = pd.DataFrame(
    [
        {
            "order_id": 105,
            "customer_id": "C2",
            "product_id": "P30",
            "order_date": "2026-03-18",
            "quantity": 2,
            "customer_name": "Bob",
            "customer_segment": "Corporate",
            "product_name": "Headphones",
            "category": "Electronics",
            "unit_price": 80.00,
        }
    ]
)

combined_df = pd.concat([merged_df, extra_orders], ignore_index=True)

# -------------------------------------------------------
# 3. Using apply() to create or transform columns
# -------------------------------------------------------
# Calculating total revenue using apply()
combined_df["total_amount"] = combined_df.apply(
    lambda row: row["quantity"] * row["unit_price"], axis=1
)


# Categorizing order scale using apply()
def categorize_order_size(total):
    if total >= 1000:
        return "High Value"
    elif total >= 200:
        return "Medium Value"
    else:
        return "Low Value"


combined_df["order_value_tier"] = combined_df["total_amount"].apply(
    categorize_order_size
)

# -------------------------------------------------------
# 4. DateTime operations: Convert date & extract components
# -------------------------------------------------------
combined_df["order_date"] = pd.to_datetime(combined_df["order_date"])

combined_df["month"] = combined_df["order_date"].dt.month_name()
combined_df["day"] = combined_df["order_date"].dt.day
combined_df["day_of_week"] = combined_df["order_date"].dt.day_name()

# -------------------------------------------------------
# 5. Organize final DataFrame & Export to CSV
# -------------------------------------------------------
column_order = [
    "order_id",
    "order_date",
    "month",
    "day",
    "day_of_week",
    "customer_id",
    "customer_name",
    "customer_segment",
    "product_id",
    "product_name",
    "category",
    "unit_price",
    "quantity",
    "total_amount",
    "order_value_tier",
]

final_processed_df = combined_df[column_order]

print("--- Final Processed Dataset Preview ---")
print(final_processed_df.to_string(index=False))

# Exporting final dataset to CSV
final_processed_df.to_csv("processed_ecommerce_dataset.csv", index=False)
print("\nDataset exported successfully to 'processed_ecommerce_dataset.csv'.")

--- Final Processed Dataset Preview ---
 order_id order_date    month  day day_of_week customer_id customer_name customer_segment product_id product_name    category  unit_price  quantity  total_amount order_value_tier
      101 2026-01-15  January   15    Thursday          C1         Alice         Consumer        P10       Laptop Electronics      1200.0         2        2400.0       High Value
      102 2026-02-20 February   20      Friday          C2           Bob        Corporate        P20        Chair   Furniture       150.0         1         150.0        Low Value
      103 2026-03-05    March    5    Thursday          C1         Alice         Consumer        P30   Headphones Electronics        80.0         4         320.0     Medium Value
      104 2026-03-12    March   12    Thursday          C3       Charlie         Consumer        P10       Laptop Electronics      1200.0         1        1200.0       High Value
      105 2026-03-18    March   18   Wednesday          C2       